# Notebook 11 — Environmental Feature Engineering

**Purpose:** Merge three district-level environmental signals — fire intensity, recoverable residue, and NASA POWER weather (temperature + rainfall) — into a single feature table for clustering and scoring.

**Missing value strategy:** Median imputation so no district is silently dropped.

**Input:** `Data/Processed/fire_stats.csv`, `Data/Processed/residue_calc.csv`, `Data/Raw/Weather/*.csv`  
**Output:** `Data/Processed/env_features.csv`

In [1]:
import pandas as pd
import numpy as np
import os

In [3]:
# Feature 1: Fire count (mean across years, per district) 
fire = pd.read_csv('../Data/Processed/fire_stats.csv')
fire_district = (
    fire.groupby('district')['fire_count']
    .mean()
    .reset_index()
)
fire_district['district'] = fire_district['district'].str.strip().str.title()

print(f"Fire feature: {len(fire_district)} districts")
print(fire_district.head())


Fire feature: 43 districts
   district    fire_count
0    Ambala    869.666667
1  Amritsar   4014.888889
2   Barnala   6303.888889
3  Bathinda  10259.777778
4   Bhiwani    219.888889


In [5]:
# Feature 2: Recoverable residue (mean across years, per district) 
residue = pd.read_csv('../Data/Processed/residue_calc.csv')

# Strip numbering prefix like '1. Punjab' if present
residue['district'] = (
    residue['district']
    .str.replace(r'^\d+\.\s*', '', regex=True)
    .str.strip().str.title()
)

residue_district = (
    residue.groupby('district')['recoverable_t']
    .mean()
    .reset_index()
    .rename(columns={'recoverable_t': 'residue'})
)

print(f"Residue feature: {len(residue_district)} districts")
print(residue_district.head())


Residue feature: 45 districts
   district        residue
0    Ambala  150972.500000
1  Amritsar  228511.500000
2   Barnala  167376.865385
3  Bathinda  312282.519231
4   Bhiwani   84831.250000


In [7]:
# Feature 3: Weather (avg temperature + total rainfall per district) 
# NASA POWER CSV files have 25 metadata rows before the data header.
WEATHER_DIR = '../Data/Raw/Weather/'
weather_records = []

for fname in os.listdir(WEATHER_DIR):
    if not fname.endswith('.csv'):
        continue

    district = fname.replace('_weather.csv', '').strip().title()
    fpath    = os.path.join(WEATHER_DIR, fname)

    try:
        df = pd.read_csv(fpath, skiprows=range(0, 25), on_bad_lines='skip')

        # Average of daily max and min temperature
        df['avg_temp'] = (df['T2M_MAX'] + df['T2M_MIN']) / 2

        weather_records.append({
            'district': district,
            'avg_temp': df['avg_temp'].mean(),
            'rainfall': df['PRECTOTCORR'].sum(),
        })
    except Exception as e:
        print(f"  ⚠  Could not read {fname}: {e}")

weather_df = pd.DataFrame(weather_records)
print(f"\nWeather feature: {len(weather_df)} districts")
print(weather_df.head())



Weather feature: 40 districts
   district   avg_temp  rainfall
0    Ambala  24.342247  11307.73
1  Amritsar  25.276999   8095.50
2   Barnala  25.451632   6914.40
3  Bathinda  26.170183   5009.70
4   Bhiwani  25.908207   6176.26


In [8]:
# Merge all three features 
# Outer join: retain every district that appears in any source.
# Districts missing from one source will have NaN — imputed next.
env = (
    fire_district
    .merge(residue_district, on='district', how='outer')
    .merge(weather_df,       on='district', how='outer')
)
print(f"Merged shape   : {env.shape}")
print(f"Missing before imputation:\n{env.isnull().sum().to_string()}")


Merged shape   : (47, 5)
Missing before imputation:
district      0
fire_count    4
residue       2
avg_temp      7
rainfall      7


In [9]:
# Median imputation 
# Median is used (not mean) because fire_count is right-skewed.
# Districts that gain imputed values are flagged so they can be noted
# in the methodology section.
imputed_districts = env[env.isnull().any(axis=1)]['district'].tolist()
if imputed_districts:
    print(f"Districts receiving median imputation: {imputed_districts}")

env.fillna(env.median(numeric_only=True), inplace=True)
print(f"\nMissing after imputation: {env.isnull().sum().sum()}")
print(f"Final shape: {env.shape}")
print(env.head())


Districts receiving median imputation: ['Charkhi Dadri', 'Fatehgarh Sahib', 'Ferozepur', 'Firozepur', 'Malerkotla', 'Mohali', 'Nawanshahr', 'S.A.S Nagar']

Missing after imputation: 0
Final shape: (47, 5)
   district    fire_count        residue   avg_temp  rainfall
0    Ambala    869.666667  150972.500000  24.342247  11307.73
1  Amritsar   4014.888889  228511.500000  25.276999   8095.50
2   Barnala   6303.888889  167376.865385  25.451632   6914.40
3  Bathinda  10259.777778  312282.519231  26.170183   5009.70
4   Bhiwani    219.888889   84831.250000  25.908207   6176.26


In [10]:
# Save 
env.to_csv('../Data/Processed/env_features.csv', index=False)
print("Saved: Data/Processed/env_features.csv")
print(f"   Rows    : {len(env)}")
print(f"   Columns : {env.columns.tolist()}")


✅ Saved: Data/Processed/env_features.csv
   Rows    : 47
   Columns : ['district', 'fire_count', 'residue', 'avg_temp', 'rainfall']
